# Fine-Tune Llama 3.1 8B for Browser Use in Google Colab
This notebook will guide you through fine-tuning Llama 3.1 8B for browser automation tasks using Unsloth, and export it for use with Ollama.

**Requirements:**
- Google Colab with GPU runtime (Free T4 or better)
- Hugging Face account and token
- ~15GB disk space

**Steps:**
1. Setup environment and install dependencies
2. Load and prepare browser use dataset
3. Fine-tune the model with Unsloth
4. Export to GGUF for Ollama
5. (Optional) Deploy on RunPod

---

## Step 1: Enable GPU Runtime

Before running any code:
1. Go to `Runtime` > `Change runtime type`
2. Select `T4 GPU` (free) or better
3. Click `Save`

Verify GPU availability:

In [ ]:
!nvidia-smi

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB" if torch.cuda.is_available() else "No GPU")

## Step 2: Install Dependencies

Unsloth provides 2x faster training and 60% less memory usage for fine-tuning LLMs.

In [ ]:
# Install Unsloth and dependencies
!pip install --upgrade --force-reinstall --no-cache-dir \
    unsloth==2024.12.4 \
    unsloth_zoo==2024.12.1 \
    torch==2.4.0 \
    triton==3.0.0 \
    xformers==0.0.27.post2 \
    transformers==4.46.3 \
    datasets==3.1.0 \
    trl==0.12.0 \
    peft==0.13.2 \
    accelerate==0.28.0 \
    bitsandbytes==0.44.1

# Install llama.cpp for GGUF conversion
!pip install llama-cpp-python==0.3.1

print("\nAll dependencies installed successfully!")

## Step 3: Configure Hugging Face

You need a Hugging Face token to download Llama 3.1 8B.

1. Create an account at https://huggingface.co
2. Go to https://huggingface.co/settings/tokens
3. Create a token with 'read' permissions
4. Accept the Llama 3.1 license at: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

In [ ]:
from huggingface_hub import login
import os

# Option 1: Use Colab secrets (recommended)
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Logged in via Colab secrets")
except:
    # Option 2: Enter token manually
    print("Please enter your Hugging Face token:")
    hf_token = input()
    login(token=hf_token)
    print("Logged in successfully!")

## Step 4: Load Llama 3.1 8B Model

We'll use 4-bit quantization to fit the model in Colab's free GPU memory.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Model settings
max_seq_length = 2048  # Context length for browser tasks
dtype = None  # Auto-detect
load_in_4bit = True  # Use 4-bit quantization for memory efficiency

# Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(f"\nModel loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e9:.2f}B")

## Step 5: Configure LoRA Adapters

Use Parameter-Efficient Fine-Tuning (PEFT) with LoRA to train efficiently.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank - higher = more parameters, better performance
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,  # LoRA scaling
    lora_dropout = 0,  # Supports any, but = 0 is optimized
    bias = "none",  # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",  # Unsloth's optimized gradient checkpointing
    random_state = 3407,
    use_rslora = False,  # Rank stabilized LoRA
    loftq_config = None,  # And LoftQ
n)

print("LoRA adapters configured!")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

## Step 6: Prepare Browser Use Dataset

Create a dataset for browser automation tasks. The model will learn to output structured actions for web navigation.

Dataset format:
```json
{
  "instruction": "Navigate to Amazon and find the best rated laptop under $800",
  "input": "Current page: google.com\nTask: Find budget laptop",
  "output": "Action: click('search_box')\nAction: type('best rated laptop under $800')\nAction: press('Enter')"
}
```

In [ ]:
import json
from datasets import Dataset

# Sample browser use training data
# In practice, you'd use a larger dataset like those from:
# - https://github.com/Khang-9966/Computer-Browser-Phone-Use-Agent-Datasets
# - https://huggingface.co/datasets/browseruse/webagent-tasks

browser_use_data = [
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Find the current weather in San Francisco\nCurrent page: google.com\nAvailable elements: search_box, search_button, result_links",
        "output": "Action: click('search_box')\nAction: type('weather San Francisco')\nAction: press('Enter')\nAction: wait(2)\nAction: read('weather_result')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Search for Python tutorials and open the first result\nCurrent page: google.com\nAvailable elements: search_box, search_button",
        "output": "Action: click('search_box')\nAction: type('Python tutorials')\nAction: press('Enter')\nAction: wait(1)\nAction: click('result_1')\nAction: wait(2)\nAction: scrollTo('bottom')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Add item to cart on e-commerce site\nCurrent page: product_page\nAvailable elements: add_to_cart_button, quantity_selector, checkout_button",
        "output": "Action: select('quantity_selector', '2')\nAction: click('add_to_cart_button')\nAction: wait(1)\nAction: click('checkout_button')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Fill out a contact form\nCurrent page: contact_form\nAvailable elements: name_field, email_field, message_field, submit_button",
        "output": "Action: click('name_field')\nAction: type('John Doe')\nAction: click('email_field')\nAction: type('john@example.com')\nAction: click('message_field')\nAction: type('Hello, I have a question about your services.')\nAction: click('submit_button')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Login to website\nCurrent page: login_page\nAvailable elements: username_field, password_field, login_button, forgot_password_link",
        "output": "Action: click('username_field')\nAction: type('myusername')\nAction: click('password_field')\nAction: type('mypassword123')\nAction: click('login_button')\nAction: wait(2)\nAction: verify('dashboard_loaded')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Navigate through pagination\nCurrent page: search_results_page_1\nAvailable elements: next_button, page_2_link, page_3_link",
        "output": "Action: click('next_button')\nAction: wait(1)\nAction: scrollTo('top')\nAction: read('page_indicator')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Compare prices of products\nCurrent page: product_listing\nAvailable elements: product_cards, price_tags, sort_dropdown",
        "output": "Action: scrollTo('middle')\nAction: read('product_1_price')\nAction: read('product_2_price')\nAction: click('sort_dropdown')\nAction: select('price_low_to_high')\nAction: wait(1)\nAction: read('sorted_results')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Download a file\nCurrent page: downloads_page\nAvailable elements: download_button, file_list, filter_input",
        "output": "Action: click('filter_input')\nAction: type('report.pdf')\nAction: press('Enter')\nAction: wait(1)\nAction: click('download_button')\nAction: wait(3)\nAction: verify('download_complete')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Close popup modal\nCurrent page: website_with_popup\nAvailable elements: close_button, accept_cookies_button, modal_overlay",
        "output": "Action: click('accept_cookies_button')\nAction: wait(0.5)\nAction: click('close_button')\nAction: verify('popup_closed')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Search and filter results\nCurrent page: ecommerce_site\nAvailable elements: search_bar, filter_category, filter_price, apply_button",
        "output": "Action: click('search_bar')\nAction: type('wireless headphones')\nAction: press('Enter')\nAction: wait(2)\nAction: click('filter_category')\nAction: select('Electronics')\nAction: click('filter_price')\nAction: type('50-100')\nAction: click('apply_button')\nAction: wait(1)\nAction: read('filtered_results')"
    },
    # Add more examples for better training
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Navigate to a specific category menu\nCurrent page: homepage\nAvailable elements: menu_toggle, electronics_category, laptops_subcategory",
        "output": "Action: hover('menu_toggle')\nAction: wait(0.5)\nAction: click('electronics_category')\nAction: wait(1)\nAction: click('laptops_subcategory')\nAction: wait(2)\nAction: scrollTo('middle')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Accept cookies and dismiss banner\nCurrent page: news_site\nAvailable elements: cookie_accept_button, newsletter_popup_close, continue_reading_link",
        "output": "Action: click('cookie_accept_button')\nAction: wait(0.3)\nAction: click('newsletter_popup_close')\nAction: click('continue_reading_link')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Switch between tabs\nCurrent page: main_tab\nAvailable elements: new_tab_button, tab_switcher, tab_list",
        "output": "Action: click('new_tab_button')\nAction: wait(1)\nAction: type('https://example.com')\nAction: press('Enter')\nAction: wait(2)\nAction: click('tab_switcher')\nAction: click('main_tab')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Scroll through infinite feed\nCurrent page: social_media\nAvailable elements: scroll_container, load_more_button, refresh_indicator",
        "output": "Action: scrollTo('bottom')\nAction: wait(1)\nAction: scrollTo('bottom')\nAction: wait(1)\nAction: scrollTo('bottom')\nAction: verify('new_content_loaded')"
    },
    {
        "instruction": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it.",
        "input": "Task: Fill multi-step form\nCurrent page: registration_step_1\nAvailable elements: next_button, personal_info_fields, progress_indicator",
        "output": "Action: fill_form({'name': 'John', 'email': 'john@example.com'})\nAction: click('next_button')\nAction: wait(1)\nAction: fill_form({'address': '123 Main St', 'city': 'New York'})\nAction: click('next_button')\nAction: wait(1)\nAction: click('submit')"
    },
]

# Create dataset
dataset = Dataset.from_list(browser_use_data)
print(f"Dataset created with {len(dataset)} examples")
print("\nSample data:")
print(json.dumps(dataset[0], indent=2))

## Step 7: Format Dataset for Training

Convert the dataset to the chat format expected by Llama 3.1.

In [ ]:
def format_prompts(examples):
    """
    Format examples in Llama 3.1 chat template:
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    {instruction}<|eot_id|><|start_header_id|>user<|end_header_id|>
    {input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
    {output}<|eot_id|>
    """
    texts = []
    for instruction, input_text, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = tokenizer.apply_chat_template([
            {"role": "system", "content": instruction},
            {"role": "user", "content": input_text},
            {"role": "assistant", "content": output},
        ], tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(format_prompts, batched=True)

print("Dataset formatted!")
print("\nSample formatted text:")
print(dataset[0]['text'][:500] + "...")

## Step 8: Fine-Tune the Model

Use Supervised Fine-Tuning (SFT) with Unsloth's optimized trainer.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_available

# Training arguments
training_args = TrainingArguments(
    per_device_train_batch_size = 2,  # Small batch size for Colab
    gradient_accumulation_steps = 4,  # Accumulate gradients
    warmup_steps = 10,
    max_steps = 60,  # Adjust based on dataset size
    learning_rate = 2e-4,  # Good starting point for LoRA
    fp16 = not is_bfloat16_available(),
    bf16 = is_bfloat16_available(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",  # Disable wandb/tensorboard
)

# Initialize trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,  # Disable packing for instruction tuning
    args = training_args,
)

# Train!
print("Starting training...")
trainer_stats = trainer.train()
print("\nTraining completed!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training samples: {trainer_stats.metrics['train_samples']}")

## Step 9: Test the Fine-Tuned Model

Test the model on a new browser task.

In [ ]:
# Enable faster inference
FastLanguageModel.for_inference(model)

# Test prompt
test_task = """Task: Find the product price and add to wishlist
Current page: product_detail
Available elements: price_tag, wishlist_button, share_button, buy_now_button"""

messages = [
    {"role": "system", "content": "You are a browser automation agent. Given a web task, output the sequence of actions to complete it."},
    {"role": "user", "content": test_task},
]

# Generate response
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("Test Task:")
print(test_task)
print("\nGenerated Actions:")
print(response.split("assistant")[-1].strip())

## Step 10: Save and Export to Ollama Format

Convert the fined-tuned model to GGUF format for use with Ollama.

In [ ]:
# Save LoRA adapters
model.save_pretrained("browser_use_lora")
tokenizer.save_pretrained("browser_use_lora")

print("LoRA adapters saved to 'browser_use_lora/'")

In [ ]:
# Merge LoRA and save full model (optional, for GGUF conversion)
model.save_pretrained_gguf(
    "browser_use_model",
    tokenizer,
    quantization_method = "q4_k_m",  # Good balance of quality and size
)

print("\nGGUF model saved to 'browser_use_model/'")
print("Files created:")
!ls -lh browser_use_model/

## Step 11: Create Ollama Modelfile

Create a Modelfile for Ollama to use the fine-tuned model.

In [ ]:
modelfile_content = '''# Browser Use Agent - Fine-tuned Llama 3.1 8B
FROM ./browser_use_model/unsloth.Q4_K_M.gguf

# Set parameters
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end_of_text|>"

# System prompt
SYSTEM You are a browser automation agent. Given a web task, output the sequence of actions to complete it. Use the format: Action: action_name('target_element'). Available actions: click, type, press, hover, scrollTo, wait, read, verify, select, fill_form.
'''

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile created!")
print("\n" + modelfile_content)

## Step 12: Download and Use with Ollama

Download the model files to use locally with Ollama.

In [ ]:
# Zip files for download
!zip -r browser_use_ollama.zip browser_use_model/ Modelfile

print("\nDownload the 'browser_use_ollama.zip' file from the files panel on the left.")
print("\nThen follow these steps locally:")
print("1. Install Ollama: https://ollama.com/download")
print("2. Extract the zip file")
print("3. Run: ollama create browser-agent -f Modelfile")
print("4. Chat with: ollama run browser-agent")

## Optional: Deploy on RunPod

To run on RunPod GPU for production:

1. Go to https://runpod.io and create an account
2. Deploy a GPU Pod (RTX 3090, A100, or H100 recommended)
3. Use the RunPod PyTorch template
4. Upload your model files:
   ```bash
   # In RunPod terminal
   curl -o browser_use_ollama.zip [your_download_url]
   unzip browser_use_ollama.zip
   ```
5. Install Ollama:
   ```bash
   curl httpsollama.com/install.sh | sh
   ollama serve &
   ollama create browser-agent -f Modelfile
   ```
6. Access via API:
   ```bash
   curl http://localhost:11434/api/generate -d '{
     "model": "browser-agent",
     "prompt": "Navigate to GitHub and search for browser-use"
   }'
   ```

In [ ]:
# Optional: Upload to Hugging Face
# Uncomment to push your model to the Hub

# model.push_to_hub_gguf(
#     "your-username/browser-use-llama-3.1-8b",
#     tokenizer,
#     quantization_method = ["q4_k_m", "q8_0", "q5_k_m"],
# )

---

## Summary

You've successfully:
1. Loaded Llama 3.1 8B with 4-bit quantization
2. Configured LoRA adapters for efficient fine-tuning
3. Created a browser use training dataset
4. Fine-tuned the model on browser automation tasks
5. Exported to GGUF format for Ollama
6. Created a Modelfile for deployment

### Next Steps

- **Expand the dataset**: Use larger datasets from [GUI Agent Datasets](https://github.com/Khang-9966/Computer-Browser-Phone-Use-Agent-Datasets)
- **Integrate with browser-use**: Use with [browser-use](https://github.com/browser-use/browser-use) library
- **Deploy on RunPod**: Use RunPod GPU for production inference
- **Test thoroughly**: Evaluate on real browser automation tasks

### Resources

- [Unsloth Documentation](https://unsloth.ai/)
- [Ollama](https://ollama.com/)
- [Browser-use Library](https://github.com/browser-use/browser-use)
- [RunPod](https://runpod.io)
- [Llama 3.1 Model Card](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B)